# Module 7.2: Targeted Predictors (Bai-Ng Method)

## Learning Objectives

By completing this notebook, you will:

1. Understand why standard PCA factors may have poor forecasting power for specific targets
2. Implement the Bai-Ng targeted predictor method using marginal correlation screening
3. Compare hard and soft thresholding approaches for variable selection
4. Use cross-validation to select optimal threshold parameters
5. Evaluate forecasting performance gains from targeting versus standard diffusion indexes

## Prerequisites

- Principal Component Analysis (Module 1)
- Diffusion index forecasting (Module 3)
- Variable selection concepts (Module 7.1)

## Estimated Time: 75 minutes

---

## Setup and Imports

We'll implement the Bai-Ng targeted predictor framework from scratch and compare it to standard PCA.

In [ ]:
# Purpose: Import libraries for targeted predictor analysis
# Key Concept: Screening predictors before factor extraction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(789)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. The Problem with Standard PCA Factors

### Motivation

Standard diffusion index forecasting:
1. Extract factors $\hat{F}$ from all predictors $X$ via PCA
2. Forecast: $y_{t+h} = \alpha + \beta' \hat{F}_t + \varepsilon_{t+h}$

**Problem**: PCA finds factors explaining variance in $X$, not necessarily predictive of $y$.

**Example**: If high-variance predictors are irrelevant for $y$, the first principal components may have poor forecasting power.

Let's create data where this problem occurs.

In [ ]:
# Purpose: Generate data with irrelevant high-variance predictors
# Key Concept: PCA can be misled by irrelevant high-variance variables

def generate_targeted_data(T=200, N=60, n_relevant=20, r_true=3, noise_std=0.3):
    """
    Generate data where only subset of predictors are relevant for target.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Total number of predictors
    n_relevant : int
        Number of predictors relevant for target
    r_true : int
        Number of true factors
    noise_std : float
        Idiosyncratic noise level
    
    Returns
    -------
    X, y, F_true, relevant_idx, irrelevant_idx
    """
    # Step 1: Generate true factors with AR(1) structure
    F_true = np.zeros((T, r_true))
    F_true[0, :] = np.random.randn(r_true)
    for t in range(1, T):
        F_true[t, :] = 0.7 * F_true[t-1, :] + np.sqrt(0.51) * np.random.randn(r_true)
    
    # Step 2: Select which predictors are relevant
    relevant_idx = np.random.choice(N, size=n_relevant, replace=False)
    irrelevant_idx = np.setdiff1d(np.arange(N), relevant_idx)
    
    # Step 3: Create loadings
    # Relevant predictors load on true factors
    Lambda = np.zeros((N, r_true))
    Lambda[relevant_idx, :] = np.random.randn(n_relevant, r_true) / np.sqrt(r_true)
    
    # Step 4: Generate X from factors + noise
    X = F_true @ Lambda.T
    
    # Step 5: Add high-variance noise factors to irrelevant predictors
    # This makes irrelevant predictors dominate PCA!
    n_irrelevant = len(irrelevant_idx)
    noise_factors = np.random.randn(T, 2)
    noise_loadings = np.random.randn(n_irrelevant, 2) * 2.0  # High loadings
    X[:, irrelevant_idx] = noise_factors @ noise_loadings.T
    
    # Add idiosyncratic noise
    X += noise_std * np.random.randn(T, N)
    
    # Step 6: Generate target variable
    # y depends only on first two factors
    beta_true = np.array([1.0, 0.5, 0.0])[:r_true]
    y = F_true @ beta_true + 0.5 * np.random.randn(T)
    
    return X, y, F_true, relevant_idx, irrelevant_idx

# Generate data
T, N = 200, 60
n_relevant = 20
X, y, F_true, relevant_idx, irrelevant_idx = generate_targeted_data(
    T=T, N=N, n_relevant=n_relevant
)

print("="*70)
print("DATA CHARACTERISTICS")
print("="*70)
print(f"Time periods (T): {T}")
print(f"Total predictors (N): {N}")
print(f"Relevant predictors: {n_relevant} ({n_relevant/N*100:.1f}%)")
print(f"Irrelevant predictors: {len(irrelevant_idx)} ({len(irrelevant_idx)/N*100:.1f}%)")
print(f"Target variable - Mean: {y.mean():.3f}, Std: {y.std():.3f}")
print("="*70)

### Demonstrate the Problem

Let's show that standard PCA extracts factors dominated by irrelevant predictors.

In [ ]:
# Purpose: Show standard PCA performance on this data
# Key Concept: Standard PCA may extract irrelevant factors

# Standardize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/test
T_train = 150
X_train, X_test = X_scaled[:T_train], X_scaled[T_train:]
y_train, y_test = y[:T_train], y[T_train:]

# Standard PCA approach
n_factors = 5
pca_standard = PCA(n_components=n_factors)
F_train_standard = pca_standard.fit_transform(X_train)
F_test_standard = pca_standard.transform(X_test)

# Forecast with 1-period horizon
horizon = 1
reg_standard = LinearRegression()
reg_standard.fit(F_train_standard[:-horizon], y_train[horizon:])
y_pred_standard = reg_standard.predict(F_test_standard[:-horizon])

mse_standard = mean_squared_error(y_test[horizon:], y_pred_standard)

# Analyze which predictors dominate first PC
pc1_loadings = pca_standard.components_[0, :]
top_10_idx = np.argsort(np.abs(pc1_loadings))[-10:]
n_relevant_in_top10 = len(set(top_10_idx) & set(relevant_idx))

print("\n" + "="*70)
print("STANDARD PCA ANALYSIS")
print("="*70)
print(f"Variance explained by first {n_factors} PCs: {pca_standard.explained_variance_ratio_.sum()*100:.1f}%")
print(f"\nFirst PC dominated by:")
print(f"  Top 10 loading predictors: {n_relevant_in_top10}/10 are relevant")
print(f"  Remaining {10-n_relevant_in_top10}/10 are irrelevant")
print(f"\nOut-of-sample forecast MSE: {mse_standard:.4f}")
print("="*70)

print("\n⚠️  Problem: PCA extracts factors dominated by irrelevant predictors!")

## 2. Targeted Predictors: Bai-Ng Method

### The Targeting Idea

**Key Insight**: Pre-select predictors based on their marginal correlation with target $y$ before extracting factors.

**Algorithm:**
1. Compute marginal correlation: $\hat{\rho}_j = \text{Corr}(X_{jt}, y_{t+h})$
2. Screen predictors based on $|\hat{\rho}_j|$
3. Extract factors from selected predictors only
4. Forecast with targeted factors

### Two Screening Approaches

**Hard Thresholding**: Select if $|\hat{\rho}_j| > c$

**Soft Thresholding**: Weight by $w_j = \frac{|\hat{\rho}_j|^\gamma}{\sum_k |\hat{\rho}_k|^\gamma}$

In [ ]:
# Purpose: Implement Targeted Predictors class
# Key Concept: Screen predictors before factor extraction

class TargetedPredictors:
    """
    Bai-Ng Targeted Predictor extraction.
    
    Parameters
    ----------
    n_factors : int
        Number of factors to extract
    threshold_type : str
        'hard' or 'soft' thresholding
    threshold : float, optional
        Threshold for hard thresholding
    gamma : float
        Exponent for soft thresholding (default: 1.0)
    horizon : int
        Forecast horizon for correlation computation
    """
    
    def __init__(self, n_factors=5, threshold_type='hard', threshold=None, 
                 gamma=1.0, horizon=1):
        self.n_factors = n_factors
        self.threshold_type = threshold_type
        self.threshold = threshold
        self.gamma = gamma
        self.horizon = horizon
        
        self.scaler = StandardScaler()
        self.pca = None
        self.reg = LinearRegression()
        
        self.correlations_ = None
        self.selected_predictors_ = None
        self.weights_ = None
        self.targeted_factors_ = None
    
    def fit(self, X, y):
        """
        Fit targeted predictor model.
        
        Parameters
        ----------
        X : array, shape (T, N)
            Predictor panel
        y : array, shape (T,)
            Target variable
        """
        X = np.asarray(X)
        y = np.asarray(y)
        T, N = X.shape
        
        # Step 1: Standardize
        X = self.scaler.fit_transform(X)
        
        # Step 2: Compute marginal correlations with target at horizon h
        T_corr = T - self.horizon
        self.correlations_ = np.zeros(N)
        
        for j in range(N):
            x_vals = X[:T_corr, j]
            y_vals = y[self.horizon:T_corr + self.horizon]
            
            if len(x_vals) > 1 and np.std(x_vals) > 0:
                self.correlations_[j], _ = pearsonr(x_vals, y_vals)
            else:
                self.correlations_[j] = 0
        
        # Handle NaN correlations
        self.correlations_ = np.nan_to_num(self.correlations_, 0)
        
        # Step 3: Screening/Weighting
        if self.threshold_type == 'hard':
            # Hard thresholding
            if self.threshold is None:
                # Auto-select: median absolute correlation
                self.threshold = np.median(np.abs(self.correlations_))
            
            self.selected_predictors_ = np.where(
                np.abs(self.correlations_) >= self.threshold
            )[0]
            
            # Ensure enough predictors
            if len(self.selected_predictors_) < self.n_factors:
                sorted_idx = np.argsort(np.abs(self.correlations_))[::-1]
                self.selected_predictors_ = sorted_idx[:max(self.n_factors, 10)]
            
            X_selected = X[:, self.selected_predictors_]
            self.weights_ = None
            
        else:  # soft thresholding
            # Compute weights
            abs_corr = np.abs(self.correlations_)
            weights = abs_corr ** self.gamma
            weights = weights / (weights.sum() + 1e-10)
            self.weights_ = weights
            
            # Weight predictors
            X_selected = X * np.sqrt(weights)
            self.selected_predictors_ = np.arange(N)
        
        # Step 4: Extract targeted factors via PCA
        self.pca = PCA(n_components=self.n_factors)
        self.targeted_factors_ = self.pca.fit_transform(X_selected)
        
        # Step 5: Forecast regression
        T_forecast = T - self.horizon
        F_t = self.targeted_factors_[:T_forecast, :]
        y_ahead = y[self.horizon:T_forecast + self.horizon]
        
        self.reg.fit(F_t, y_ahead)
        
        return self
    
    def predict(self, X):
        """
        Generate h-step ahead forecasts.
        
        Parameters
        ----------
        X : array, shape (T_new, N)
            New predictor data
        
        Returns
        -------
        forecasts : array
        """
        X = np.asarray(X)
        X = self.scaler.transform(X)
        
        # Apply targeting
        if self.threshold_type == 'hard':
            X_selected = X[:, self.selected_predictors_]
        else:
            X_selected = X * np.sqrt(self.weights_)
        
        # Transform to factors
        factors = self.pca.transform(X_selected)
        
        # Forecast
        forecasts = self.reg.predict(factors)
        
        return forecasts

print("TargetedPredictors class implemented!")

## 3. Comparing Standard vs Targeted Factors

Let's apply the targeted predictor method and compare to standard PCA.

In [ ]:
# Purpose: Compare standard PCA vs targeted predictors
# Key Concept: Targeting should improve forecast accuracy

# Fit targeted predictors with hard thresholding
print("Fitting Targeted Predictors (hard thresholding)...")
target_hard = TargetedPredictors(
    n_factors=5, 
    threshold_type='hard', 
    threshold=0.1, 
    horizon=1
)
target_hard.fit(X_train, y_train)
y_pred_hard = target_hard.predict(X_test[:-1])
mse_hard = mean_squared_error(y_test[1:], y_pred_hard)

# Fit targeted predictors with soft thresholding
print("Fitting Targeted Predictors (soft thresholding, γ=2)...")
target_soft = TargetedPredictors(
    n_factors=5, 
    threshold_type='soft', 
    gamma=2.0, 
    horizon=1
)
target_soft.fit(X_train, y_train)
y_pred_soft = target_soft.predict(X_test[:-1])
mse_soft = mean_squared_error(y_test[1:], y_pred_soft)

# Summary
print("\n" + "="*70)
print("STANDARD vs TARGETED PREDICTORS")
print("="*70)
print(f"Data: T={T}, N={N} ({n_relevant} relevant predictors)\n")

print("Out-of-Sample MSE:")
print(f"  Standard PCA:           {mse_standard:.4f}")
print(f"  Targeted (Hard):        {mse_hard:.4f}  ({(mse_standard-mse_hard)/mse_standard*100:+.1f}%)")
print(f"  Targeted (Soft, γ=2):   {mse_soft:.4f}  ({(mse_standard-mse_soft)/mse_standard*100:+.1f}%)\n")

print(f"Hard thresholding selected: {len(target_hard.selected_predictors_)} predictors")
n_recovered = len(set(target_hard.selected_predictors_) & set(relevant_idx))
print(f"  True relevant recovered: {n_recovered}/{n_relevant} ({n_recovered/n_relevant*100:.1f}%)")
print("="*70)

# Determine improvement
best_mse = min(mse_hard, mse_soft)
improvement = (mse_standard - best_mse) / mse_standard * 100
print(f"\n✓ Best targeting method improves MSE by {improvement:.1f}%")

### Visualizing Targeting Results

In [ ]:
# Purpose: Visualize targeting analysis
# Key Concept: Targeting identifies relevant predictors

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Distribution of correlations
axes[0, 0].hist(target_hard.correlations_, bins=30, alpha=0.7, 
               color='steelblue', edgecolor='black')
axes[0, 0].axvline(target_hard.threshold, color='red', linestyle='--', 
                   linewidth=2, label=f'Threshold={target_hard.threshold:.3f}')
axes[0, 0].axvline(-target_hard.threshold, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Marginal Correlation with Target', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Distribution of Predictor Correlations', 
                     fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Sorted absolute correlations
sorted_abs_corr = np.sort(np.abs(target_hard.correlations_))[::-1]
axes[0, 1].plot(sorted_abs_corr, linewidth=2, color='steelblue')
n_selected = len(target_hard.selected_predictors_)
axes[0, 1].axvline(n_selected, color='red', linestyle='--', linewidth=2,
                   label=f'{n_selected} selected')
axes[0, 1].axhline(target_hard.threshold, color='red', linestyle='--', 
                   alpha=0.5, linewidth=2)
axes[0, 1].set_xlabel('Predictor Rank', fontsize=11)
axes[0, 1].set_ylabel('Absolute Correlation', fontsize=11)
axes[0, 1].set_title('Sorted Absolute Correlations', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Variance explained by factors
var_explained_standard = pca_standard.explained_variance_ratio_
var_explained_targeted = target_hard.pca.explained_variance_ratio_

x = np.arange(1, n_factors + 1)
width = 0.35
axes[1, 0].bar(x - width/2, var_explained_standard, width, 
              label='Standard PCA', alpha=0.7, color='coral', edgecolor='black')
axes[1, 0].bar(x + width/2, var_explained_targeted, width, 
              label='Targeted', alpha=0.7, color='seagreen', edgecolor='black')
axes[1, 0].set_xlabel('Factor Number', fontsize=11)
axes[1, 0].set_ylabel('Variance Explained', fontsize=11)
axes[1, 0].set_title('Variance Explained by Factors', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Out-of-sample predictions
axes[1, 1].plot(y_test[1:], 'k-', linewidth=2, label='Actual', alpha=0.7)
axes[1, 1].plot(y_pred_standard, '--', linewidth=2, label='Standard PCA', alpha=0.7)
axes[1, 1].plot(y_pred_hard, ':', linewidth=2, label='Targeted (Hard)', alpha=0.7)
axes[1, 1].set_xlabel('Time Period', fontsize=11)
axes[1, 1].set_ylabel('Target Variable', fontsize=11)
axes[1, 1].set_title('Out-of-Sample Forecasts', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Cross-Validation for Threshold Selection

The threshold parameter $c$ (for hard thresholding) or $\gamma$ (for soft thresholding) should be selected via cross-validation to optimize out-of-sample performance.

In [ ]:
# Purpose: Cross-validation for threshold selection
# Key Concept: Optimal threshold minimizes CV forecast error

def cv_threshold_selection(X, y, n_factors=5, cv_folds=5, horizon=1):
    """
    Select optimal threshold via time-series cross-validation.
    
    Parameters
    ----------
    X : array, shape (T, N)
        Predictors
    y : array, shape (T,)
        Target
    n_factors : int
        Number of factors
    cv_folds : int
        Number of CV folds
    horizon : int
        Forecast horizon
    
    Returns
    -------
    results_df : DataFrame
        CV results for different thresholds
    """
    T = len(y)
    
    # Compute all correlations first to generate threshold candidates
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    all_corr = []
    for j in range(X.shape[1]):
        if T > horizon:
            corr, _ = pearsonr(X_scaled[:-horizon, j], y[horizon:])
            all_corr.append(abs(corr))
    
    all_corr = np.array(all_corr)
    thresholds = np.percentile(all_corr, [10, 20, 30, 40, 50, 60, 70, 80, 90])
    
    # Time-series CV
    tscv = TimeSeriesSplit(n_splits=cv_folds)
    results = []
    
    for threshold in thresholds:
        fold_errors = []
        
        for train_idx, val_idx in tscv.split(X):
            X_train_cv = X[train_idx]
            y_train_cv = y[train_idx]
            X_val_cv = X[val_idx]
            y_val_cv = y[val_idx]
            
            # Fit targeted model
            model = TargetedPredictors(
                n_factors=n_factors,
                threshold_type='hard',
                threshold=threshold,
                horizon=horizon
            )
            model.fit(X_train_cv, y_train_cv)
            
            # Predict on validation
            if len(X_val_cv) > horizon:
                y_pred = model.predict(X_val_cv[:-horizon])
                error = mean_squared_error(y_val_cv[horizon:], y_pred)
                fold_errors.append(error)
        
        if fold_errors:
            results.append({
                'threshold': threshold,
                'mean_mse': np.mean(fold_errors),
                'std_mse': np.std(fold_errors)
            })
    
    results_df = pd.DataFrame(results)
    return results_df

# Run CV
print("Running cross-validation for threshold selection...")
cv_results = cv_threshold_selection(X_train, y_train, n_factors=5, cv_folds=5, horizon=1)

# Find best threshold
best_threshold = cv_results.loc[cv_results['mean_mse'].idxmin(), 'threshold']

print("\n" + "="*70)
print("CROSS-VALIDATION RESULTS")
print("="*70)
print(cv_results.to_string(index=False))
print(f"\nOptimal threshold: {best_threshold:.4f}")
print("="*70)

### Visualize CV Results

In [ ]:
# Purpose: Plot cross-validation results
# Key Concept: CV curve shows bias-variance tradeoff

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(cv_results['threshold'], cv_results['mean_mse'], 
       'b-o', linewidth=2, markersize=8, label='Mean CV MSE')
ax.fill_between(cv_results['threshold'],
               cv_results['mean_mse'] - cv_results['std_mse'],
               cv_results['mean_mse'] + cv_results['std_mse'],
               alpha=0.3, color='blue', label='±1 Std Dev')

# Mark optimal threshold
ax.axvline(best_threshold, color='red', linestyle='--', linewidth=2,
          label=f'Optimal = {best_threshold:.3f}')

ax.set_xlabel('Threshold (c)', fontsize=12)
ax.set_ylabel('Cross-Validation MSE', fontsize=12)
ax.set_title('Threshold Selection via Cross-Validation', 
            fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Low threshold: Include many predictors (high variance)")
print("- High threshold: Include few predictors (high bias)")
print("- Optimal threshold balances bias-variance tradeoff")

## 5. Exercises

### Exercise 5.1: Implement Soft Thresholding Weight Function

**Task:** Create a function that computes soft thresholding weights for different $\gamma$ values and visualize how they concentrate mass on high-correlation predictors.

In [ ]:
# Exercise 5.1: Soft thresholding weights

def compute_soft_weights(correlations, gamma):
    """
    Compute soft thresholding weights.
    
    Parameters
    ----------
    correlations : array
        Marginal correlations
    gamma : float
        Exponent parameter
    
    Returns
    -------
    weights : array
        Normalized weights summing to 1
    """
    # YOUR CODE HERE
    # Hint: weights = |correlation|^gamma / sum(|correlation|^gamma)
    # ---------------
    
    weights = None  # Replace with your implementation
    
    return weights

# Visualize for different gamma values
test_corr = target_hard.correlations_
sorted_idx = np.argsort(np.abs(test_corr))[::-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
gamma_values = [0.5, 1.0, 2.0]

for i, gamma in enumerate(gamma_values):
    weights = compute_soft_weights(test_corr, gamma)
    sorted_weights = weights[sorted_idx]
    
    axes[i].bar(range(N), sorted_weights, alpha=0.7, edgecolor='black')
    axes[i].set_xlabel('Predictor Rank', fontsize=11)
    axes[i].set_ylabel('Weight', fontsize=11)
    axes[i].set_title(f'γ = {gamma}', fontsize=12, fontweight='bold')
    axes[i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# AUTO-GRADED TESTS
def test_exercise_5_1():
    assert compute_soft_weights is not None, "Implement compute_soft_weights"
    
    test_corr = np.array([0.5, 0.3, 0.1, -0.4])
    weights = compute_soft_weights(test_corr, gamma=1.0)
    
    assert weights is not None, "Function should return weights"
    assert len(weights) == len(test_corr), "Weight array should match input length"
    assert np.isclose(weights.sum(), 1.0), "Weights should sum to 1"
    assert np.all(weights >= 0), "All weights should be non-negative"
    
    # Higher gamma should concentrate weights more
    weights_low = compute_soft_weights(test_corr, gamma=0.5)
    weights_high = compute_soft_weights(test_corr, gamma=2.0)
    
    # Maximum weight should increase with gamma
    assert weights_high.max() > weights_low.max(), "Higher gamma should concentrate weights"
    
    print("✅ Exercise 5.1 passed!")

test_exercise_5_1()

### Exercise 5.2: Compare Targeting at Different Horizons

**Task:** Investigate how targeting effectiveness changes with forecast horizon $h \in \{1, 2, 4, 8\}$.

In [ ]:
# Exercise 5.2: Targeting at different horizons

horizons = [1, 2, 4, 8]

# YOUR CODE HERE
# ---------------
# For each horizon:
#   1. Fit standard PCA
#   2. Fit targeted predictors
#   3. Compute test MSE for both
#   4. Store improvement percentage

results_by_horizon = []  # Replace with your implementation

# Create comparison plot
# YOUR CODE: Plot MSE vs horizon for both methods

# AUTO-GRADED TESTS
def test_exercise_5_2():
    assert len(results_by_horizon) == len(horizons), "Should have results for all horizons"
    
    for result in results_by_horizon:
        assert 'horizon' in result, "Each result should have 'horizon'"
        assert 'standard_mse' in result, "Each result should have 'standard_mse'"
        assert 'targeted_mse' in result, "Each result should have 'targeted_mse'"
    
    print("✅ Exercise 5.2 passed!")

test_exercise_5_2()

### Exercise 5.3: Economic Interpretation

**Task:** For the selected predictors, create a summary showing:
1. Which predictors were selected
2. Their correlations with the target
3. Whether they are truly relevant (ground truth)
4. Factor loadings on the first targeted factor

In [ ]:
# Exercise 5.3: Interpret selected predictors

# YOUR CODE HERE
# ---------------
# Create a DataFrame with columns:
#   - predictor_idx
#   - correlation
#   - truly_relevant (boolean)
#   - loading_factor1
# Sort by absolute correlation

interpretation_df = None  # Replace with your implementation

# Print summary statistics
# YOUR CODE: Print precision, recall, etc.

# AUTO-GRADED TESTS
def test_exercise_5_3():
    assert interpretation_df is not None, "Create interpretation_df"
    assert 'predictor_idx' in interpretation_df.columns, "Should have predictor_idx"
    assert 'correlation' in interpretation_df.columns, "Should have correlation"
    assert 'truly_relevant' in interpretation_df.columns, "Should have truly_relevant"
    
    # Check that selected predictors are included
    assert len(interpretation_df) > 0, "Should have at least one row"
    
    print("✅ Exercise 5.3 passed!")

test_exercise_5_3()

## 6. Summary

### Key Takeaways

1. **Standard PCA can fail for forecasting** when high-variance predictors are irrelevant for the target

2. **Targeted predictors pre-screen** based on marginal correlation with target before factor extraction

3. **Hard thresholding** selects predictors exceeding correlation cutoff (discrete selection)

4. **Soft thresholding** weights predictors by correlation strength (smooth selection)

5. **Cross-validation** is essential for selecting threshold parameters

6. **Targeting improves forecasts** when many predictors are irrelevant or when relevant predictors have low variance

### When Targeting Helps Most

- Many predictors unrelated to target
- High-variance predictors are irrelevant
- True factor structure is sparse for target
- Large predictor panels (N >> T)

### Next Steps

Next, we'll explore **three-pass regression filter** (Kelly-Pruitt), which further refines targeting by combining it with cross-sectional regression insights.

---

## Additional Resources

**Foundational Papers:**
- Bai, J. & Ng, S. (2008). "Forecasting Economic Time Series Using Targeted Predictors." *Journal of Econometrics*
- Bai, J. & Ng, S. (2009). "Boosting Diffusion Indices." *Journal of Applied Econometrics*

**Extensions:**
- Kelly, B. & Pruitt, S. (2015). "The Three-Pass Regression Filter." *Journal of Econometrics*